In [8]:
"""
DSC2107: Data Mining & Wrangling
Group 4: Kilimall Kenya Scraper (kilimall.co.ke)
Requirements: >= 500 products across 3 categories, 7-day time series.
"""
import requests
from bs4 import BeautifulSoup

url = "https://www.kilimall.co.ke/search?q=phone&page=1"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

resp = requests.get(url, headers=headers)
print("Status Code:", resp.status_code)
print("Page length:", len(resp.text))

# Check if Samsung Galaxy A06 (from your screenshot) is in the HTML text!
if "Galaxy A06" in resp.text:
    print("🎉 SUCCESS: The products are directly in the HTML! We can scrape it using BeautifulSoup!")
else:
    print("⚡ The page uses dynamic JavaScript. We will use the network endpoint.")

Status Code: 200
Page length: 286515
🎉 SUCCESS: The products are directly in the HTML! We can scrape it using BeautifulSoup!


In [9]:
import re
import requests
from bs4 import BeautifulSoup

url = "https://www.kilimall.co.ke/search?q=phone&page=1"
headers = {
    "User-Agent": "DSC2107-Course-Project (student@ucu.ac.ug)"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

# Find all links pointing to goods/products
goods_links = soup.find_all("a", href=lambda h: h and "/new/goods/" in h)

print(f"Total product links discovered: {len(goods_links)}")

if goods_links:
    # Look at the first product card
    sample_card = goods_links[0]
    print("\n--- SAMPLE PRODUCT CARD FOUND ---")
    print("Product Link:", sample_card.get("href"))
    print("Raw Card Text:\n", sample_card.get_text(separator=" | ", strip=True))
else:
    # Alternative check: find elements containing 'KSh'
    print("\nSearching for price tags containing 'KSh'...")
    ksh_tags = soup.find_all(string=re.compile(r"KSh", re.I))
    print(f"Found {len(ksh_tags)} price mentions!")
    for tag in ksh_tags[:5]:
        print("-> Price text:", tag.strip(), "| Parent tag:", tag.parent.name, "| Classes:", tag.parent.get("class"))

Total product links discovered: 0

Searching for price tags containing 'KSh'...
Found 37 price mentions!
-> Price text: KSh 13,999 | Parent tag: div | Classes: ['product-price']
-> Price text: KSh 16,499 | Parent tag: div | Classes: ['product-price']
-> Price text: KSh 20,899 | Parent tag: div | Classes: ['product-price']
-> Price text: KSh 19,599 | Parent tag: div | Classes: ['product-price']
-> Price text: KSh 10,999 | Parent tag: div | Classes: ['product-price']


In [10]:
# Grab the very first price tag
first_price_tag = ksh_tags[0]

# Find the entire card containing this product
# We walk up to the main card container
card = first_price_tag.find_parent("div", class_=lambda c: c and any("product" in x or "goods" in x or "item" in x for x in c))
if not card:
    card = first_price_tag.parent.parent  # Walk two levels up

print("=== ENTIRE PRODUCT CARD TEXT ===")
print(card.get_text(separator=" | ", strip=True))

print("\n=== ALL HTML TAGS & CLASSES INSIDE THIS CARD ===")
for elem in card.find_all(True):
    classes = elem.get("class")
    if classes:
        print(f"Tag: <{elem.name}> | Classes: {classes} | Text snippet: {elem.text[:30].strip()}")

=== ENTIRE PRODUCT CARD TEXT ===
Brand New Samsung Galaxy A06 64GB+4GB 50MP Camera 6.7" 90Hz MTK Helio G85 (12nm) 5000mAh 25W Type-C Charge Android 14 Smart Phones phone Samsung A06 | KSh 13,999 | (3254)

=== ALL HTML TAGS & CLASSES INSIDE THIS CARD ===
Tag: <p> | Classes: ['product-title'] | Text snippet: Brand New Samsung Galaxy A06
Tag: <img> | Classes: ['choice-tag'] | Text snippet: 
Tag: <div> | Classes: ['product-price'] | Text snippet: KSh 13,999
Tag: <div> | Classes: ['rate'] | Text snippet: (3254)
Tag: <div> | Classes: ['van-rate'] | Text snippet: 
Tag: <div> | Classes: ['van-rate__item'] | Text snippet: 
Tag: <i> | Classes: ['van-badge__wrapper', 'van-icon', 'van-icon-star', 'van-rate__icon', 'van-rate__icon--full'] | Text snippet: 
Tag: <div> | Classes: ['van-rate__item'] | Text snippet: 
Tag: <i> | Classes: ['van-badge__wrapper', 'van-icon', 'van-icon-star', 'van-rate__icon', 'van-rate__icon--full'] | Text snippet: 
Tag: <div> | Classes: ['van-rate__item'] | Text snippet: 


In [11]:
import sqlite3
import pandas as pd

DB_NAME = "kilimall_data.db"
CSV_PATH = "data/raw/kilimall_raw_20261013.csv"

# Connect to the existing database file
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# 1. Drop existing tables to clear the old column names
cursor.execute("DROP TABLE IF EXISTS observations")
cursor.execute("DROP TABLE IF EXISTS products")

# 2. Re-create the clean schema
cursor.execute("""
CREATE TABLE products (
    product_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    product_url TEXT
)
""")

cursor.execute("""
CREATE TABLE observations (
    observation_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id TEXT NOT NULL,
    price REAL NOT NULL,
    original_price REAL,
    discount_percent REAL,
    rating REAL,
    units_sold INTEGER,
    scraped_at TEXT NOT NULL,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
)
""")
conn.commit()

# 3. Load all 540 products from our saved CSV
df = pd.read_csv(CSV_PATH)
print(f"Loading {len(df)} products from {CSV_PATH} into SQLite...")

for _, row in df.iterrows():
    cursor.execute("""
        INSERT OR IGNORE INTO products (product_id, name, category, product_url)
        VALUES (?, ?, ?, ?)
    """, (str(row["product_id"]), str(row["name"]), str(row["category"]), str(row["product_url"])))
    
    cursor.execute("""
        INSERT INTO observations (product_id, price, original_price, discount_percent, rating, units_sold, scraped_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        str(row["product_id"]),
        float(row["price"]),
        float(row["original_price"]),
        float(row["discount_percent"]),
        float(row["rating"]),
        int(row["units_sold"]),
        str(row["scraped_at"])
    ))

conn.commit()

# 4. Verify counts
cursor.execute("SELECT COUNT(*) FROM products")
total_products = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM observations")
total_observations = cursor.fetchone()[0]

# Always close the connection
conn.close()

print("\n🎉 SUCCESS!")
print(f"✅ Total Unique Products in DB: {total_products}")
print(f"✅ Total Observations in DB: {total_observations}")

Loading 396 products from data/raw/kilimall_raw_20261013.csv into SQLite...

🎉 SUCCESS!
✅ Total Unique Products in DB: 395
✅ Total Observations in DB: 396


In [1]:
"""
=============================================================================
Course: DSC2107 - Data Mining and Wrangling
Group: Group 4
Target: Kilimall Kenya (kilimall.co.ke)
Function: Unified Daily Scraper & SQLite Storage Pipeline
Compliance: Rules of Engagement (Section 4) & Schema Specification (Part B)
=============================================================================
"""

from datetime import datetime
import json
import os
import random
import re
import sqlite3
import time
from urllib.parse import urljoin
import pandas as pd
import requests
from bs4 import BeautifulSoup

# =====================================================================
# 1. CONFIGURATION & ETHICAL HEADERS (Section 4 Compliance)
# =====================================================================
HEADERS = {
    # Section 4(c): User-Agent identifying our course and contact email
    "User-Agent": "DSC2107-Course-Project (student_email@ucu.ac.ug)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9"
}

BASE_SEARCH_URL = "https://www.kilimall.co.ke/search"

# Section 3(c): 3 distinct categories, target = 180 each -> 540 items (> 500)
CATEGORIES = [
    {"name": "Phones & Accessories", "query": "phone", "target_items": 180},
    {"name": "Audio & Electronics", "query": "audio", "target_items": 180},
    {"name": "Computers & Laptops", "query": "laptops", "target_items": 180}
]

DB_NAME = "kilimall_data.db"
RAW_DATA_DIR = os.path.join("data", "raw")

# =====================================================================
# 2. DATABASE INITIALIZATION (Part B: Relational SQLite Storage)
# =====================================================================
def init_database():
    """Initializes the normalized SQLite database schema."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Dimension Table: Static product details (one row per product)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS products (
        product_id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        category TEXT NOT NULL,
        product_url TEXT
    )
    """)
    
    # Fact Table: Time-series observations (one row per scrape run)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS observations (
        observation_id INTEGER PRIMARY KEY AUTOINCREMENT,
        product_id TEXT NOT NULL,
        price REAL NOT NULL,
        original_price REAL,
        discount_percent REAL,
        rating REAL,
        units_sold INTEGER,
        scraped_at TEXT NOT NULL,
        FOREIGN KEY (product_id) REFERENCES products(product_id)
    )
    """)
    conn.commit()
    conn.close()

# =====================================================================
# 3. PRODUCT PARSER HELPER
# =====================================================================
def parse_product_card(card, category_name, timestamp):
    """Extracts all mandatory attributes from a single product DOM container."""
    try:
        # 1. Product Name
        title_tag = card.find("p", class_="product-title")
        if not title_tag:
            return None
        title = title_tag.text.strip()

        # 2. Price (e.g. 'KSh 13,999' -> 13999.0)
        price_tag = card.find("div", class_="product-price")
        if not price_tag:
            return None
        price_clean = re.sub(r"[^\d.]", "", price_tag.text)
        if not price_clean:
            return None
        price = float(price_clean)

        # 3. Rating (Count full stars inside the van-rate component)
        full_stars = card.find_all(class_=re.compile(r"van-rate__icon--full"))
        half_stars = card.find_all(class_=re.compile(r"van-rate__icon--half"))
        rating = float(len(full_stars) + (0.5 * len(half_stars)))

        # 4. Units Sold / Reviews (e.g. '(3254)' -> 3254)
        reviews_tag = card.find("span", class_="reviews")
        if reviews_tag:
            review_clean = re.sub(r"[^\d]", "", reviews_tag.text)
            units_sold = int(review_clean) if review_clean else 0
        else:
            units_sold = 0

        # 5. Old Price & Discount %
        old_price_tag = card.find(class_=re.compile(r"origin|old|market", re.I))
        if old_price_tag:
            old_price_clean = re.sub(r"[^\d.]", "", old_price_tag.text)
            old_price = float(old_price_clean) if old_price_clean else price
        else:
            old_price = price

        if old_price > price and old_price > 0:
            discount_pct = round(((old_price - price) / old_price) * 100, 2)
        else:
            discount_pct = 0.0

        # 6. Product Link & ID
        link_tag = card.find("a") or card.find_parent("a")
        if link_tag and link_tag.get("href"):
            raw_url = link_tag.get("href")
            product_url = urljoin("https://www.kilimall.co.ke", raw_url)
            id_match = re.search(r"(\d{5,})", raw_url)
            product_id = id_match.group(1) if id_match else f"sku_{abs(hash(title)) % 10000000}"
        else:
            product_id = f"sku_{abs(hash(title)) % 10000000}"
            product_url = f"https://www.kilimall.co.ke/search?q={title[:20]}"

        return {
            "product_id": str(product_id),
            "name": title,
            "category": category_name,
            "price": price,
            "original_price": old_price,
            "discount_percent": discount_pct,
            "rating": rating,
            "units_sold": units_sold,
            "product_url": product_url,
            "scraped_at": timestamp
        }
    except Exception:
        return None

# =====================================================================
# 4. MAIN PIPELINE EXECUTION
# =====================================================================
def run_daily_collection():
    init_database()
    run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    date_stamp = datetime.now().strftime("%Y%m%d")
    
    all_products = []
    print(f"==================================================")
    print(f"Starting Kilimall Scraping Session: {run_timestamp}")
    print(f"==================================================")

    for cat in CATEGORIES:
        category_name = cat["name"]
        query = cat["query"]
        target = cat["target_items"]
        
        print(f"\n📂 Scraping: '{category_name}' (Target: {target} items)")
        collected_in_cat = 0
        page = 1
        
        while collected_in_cat < target:
            page_url = f"{BASE_SEARCH_URL}?q={query}&page={page}"
            print(f"   Fetching Page {page}: {page_url}")
            
            try:
                resp = requests.get(page_url, headers=HEADERS, timeout=15)
                if resp.status_code != 200:
                    print(f"   [Notice] Status {resp.status_code}. Stopping category.")
                    break
            except Exception as e:
                print(f"   [Error] Network error: {e}")
                break

            soup = BeautifulSoup(resp.text, "html.parser")
            title_tags = soup.find_all("p", class_="product-title")
            
            if not title_tags:
                print(f"   No products found on page {page}. Ending category.")
                break

            added_this_page = 0
            for t_tag in title_tags:
                card_container = t_tag.find_parent("div", class_=lambda c: c and any(x in str(c) for x in ["product", "item", "goods", "box"]))
                if not card_container:
                    card_container = t_tag.parent.parent
                
                item = parse_product_card(card_container, category_name, run_timestamp)
                if item:
                    all_products.append(item)
                    collected_in_cat += 1
                    added_this_page += 1
                    if collected_in_cat >= target:
                        break

            print(f"   -> Page {page}: Added {added_this_page} items. (Category Progress: {collected_in_cat}/{target})")
            page += 1

            # Section 4(b): Mandatory polite delay between 2 and 4.5 seconds
            time.sleep(random.uniform(2.0, 4.5))

    print(f"\n==================================================")
    print(f"Session Complete: Collected {len(all_products)} total products.")
    print(f"==================================================")

    if not all_products:
        print("Warning: Zero products collected.")
        return

    # =====================================================================
    # 5. STORAGE & DATABASE LOADING (Part B Compliance)
    # =====================================================================
    os.makedirs(RAW_DATA_DIR, exist_ok=True)
    
    # 1. Save Raw Evidence Files
    raw_csv_path = os.path.join(RAW_DATA_DIR, f"kilimall_raw_{date_stamp}.csv")
    raw_json_path = os.path.join(RAW_DATA_DIR, f"kilimall_raw_{date_stamp}.json")
    
    df = pd.DataFrame(all_products)
    df.to_csv(raw_csv_path, index=False)
    
    with open(raw_json_path, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    print(f"📁 Raw evidence successfully saved:")
    print(f"   - CSV:  {raw_csv_path}")
    print(f"   - JSON: {raw_json_path}")

    # 2. Ingest into Relational SQLite Database
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    for item in all_products:
        cursor.execute("""
            INSERT OR IGNORE INTO products (product_id, name, category, product_url)
            VALUES (?, ?, ?, ?)
        """, (item["product_id"], item["name"], item["category"], item["product_url"]))
        
        cursor.execute("""
            INSERT INTO observations (product_id, price, original_price, discount_percent, rating, units_sold, scraped_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            item["product_id"],
            item["price"],
            item["original_price"],
            item["discount_percent"],
            item["rating"],
            item["units_sold"],
            item["scraped_at"]
        ))
        
    conn.commit()
    
    # Print status verification
    cursor.execute("SELECT COUNT(*) FROM products")
    unique_products = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM observations")
    total_observations = cursor.fetchone()[0]
    conn.close()

    print(f"\n💾 SQLite Database Updated ('{DB_NAME}'):")
    print(f"   - Unique Products in 'products' table:     {unique_products}")
    print(f"   - Total Records in 'observations' table:    {total_observations}")

if __name__ == "__main__":
    run_daily_collection()

Starting Kilimall Scraping Session: 2026-10-13 16:50:22

📂 Scraping: 'Phones & Accessories' (Target: 180 items)
   Fetching Page 1: https://www.kilimall.co.ke/search?q=phone&page=1
   -> Page 1: Added 36 items. (Category Progress: 36/180)
   Fetching Page 2: https://www.kilimall.co.ke/search?q=phone&page=2
   -> Page 2: Added 36 items. (Category Progress: 72/180)
   Fetching Page 3: https://www.kilimall.co.ke/search?q=phone&page=3
   -> Page 3: Added 36 items. (Category Progress: 108/180)
   Fetching Page 4: https://www.kilimall.co.ke/search?q=phone&page=4
   -> Page 4: Added 36 items. (Category Progress: 144/180)
   Fetching Page 5: https://www.kilimall.co.ke/search?q=phone&page=5
   -> Page 5: Added 36 items. (Category Progress: 180/180)

📂 Scraping: 'Audio & Electronics' (Target: 180 items)
   Fetching Page 1: https://www.kilimall.co.ke/search?q=audio&page=1
   -> Page 1: Added 36 items. (Category Progress: 36/180)
   Fetching Page 2: https://www.kilimall.co.ke/search?q=audio&page=2